# UGRP · ACT 입력 비교 학습
해상도 128/256 × 이력 1/4 × seed 2개. 기존 데이터·8000 steps·batch 32를 고정한다.

Chrome **강 / kcm0127@gmail.com**에서 실행한다. **런타임 → 런타임 유형 변경 → T4 GPU**를 선택한다.
Drive를 마운트하지 않는다. 아래 셀은 [임시 노트북](https://colab.research.google.com/notebooks/empty.ipynb)에 복사해 실행할 수 있다.
노트북·데이터·결과의 Drive 저장은 이 절차에 포함되지 않는다.

로컬에서 커밋한 소스로 `python3 scripts/colab_carry_bundle.py pack --output outputs/colab/carry.zip`를 실행한 뒤 ZIP과 옆의 manifest를 준비한다.


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, json, subprocess, sys, zipfile
uploaded = files.upload()  # carry.zip + carry.manifest.json
archive = next(Path(n) for n in uploaded if n.endswith('.zip'))
record = next(json.loads(v) for n, v in uploaded.items() if n.endswith('.manifest.json'))
assert hashlib.sha256(archive.read_bytes()).hexdigest() == record['sha256']
# Bootstrap only the verified bundle utility. It validates all member paths/hashes.
with zipfile.ZipFile(archive) as z:
    bootstrap = Path('/content/colab_carry_bundle.py')
    bootstrap.write_bytes(z.read('source/scripts/colab_carry_bundle.py'))
subprocess.run([sys.executable, str(bootstrap), 'unpack', '--archive', str(archive),
                '--output', '/content/ugrp-carry', '--sha256', record['sha256']], check=True)
del uploaded


In [ ]:
# Separate research environment; do not replace Colab's notebook kernel packages.
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
subprocess.run(['uv', 'venv', '--python', '3.12', '/content/act-env'], check=True)
PY = '/content/act-env/bin/python'
SOURCE = Path('/content/ugrp-carry/source')
DATA = '/content/ugrp-carry/data/dataset.json'
subprocess.run(['uv', 'pip', 'install', '--python', PY, '-r', str(SOURCE/'requirements-reference-act.txt')], check=True)
# Same two device-only upstream fixes already used by the local ACT environment.
subprocess.run([PY, str(SOURCE/'scripts/patch_reference_act.py')], check=True)
subprocess.run([PY, '-c', 'import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name())'], check=True)


In [ ]:
# Two updates of the largest arm: cache verification + checkpoint, not a full result.
subprocess.run([PY, str(SOURCE/'scripts/ugrp_session.py'), 'run', 'colab-carry-diagnostic', '--', PY, str(SOURCE/'scripts/run_colab_carry_training.py'), '--dataset', DATA,
                '--out', '/content/carry-diagnostic', '--diagnostic'], check=True)


In [ ]:
# Full fixed cohort. If interrupted, use the same directory and add --resume.
subprocess.run([PY, str(SOURCE/'scripts/ugrp_session.py'), 'run', 'colab-carry-cohort', '--', PY, str(SOURCE/'scripts/run_colab_carry_training.py'), '--dataset', DATA,
                '--out', '/content/carry-training'], check=True)


In [ ]:
# Download while VM is alive. VM deletion removes unsaved files, including resume.pt.
import shutil
archive = shutil.make_archive('/content/carry-results', 'zip', '/content', 'carry-training')
files.download(archive)


## 중단·재개와 평가
`resume.pt`에는 optimizer, scheduler, sampling generator, CPU/CUDA RNG, 최선 가중치가 저장된다(1/500 step 및 진단 종료).
세션이 끊겨도 VM이 남아 있으면 `--resume`으로 재개한다. VM을 잃기 전 결과 ZIP을 내려받아야 한다.
새 VM에서는 데이터 ZIP을 다시 풀고, 결과 ZIP도 `/content/carry-training`으로 복원한다. 소스·데이터·설정·환경이 다르면 재개를 거부한다.

`report.json.complete`와 모델 해시를 확인한 뒤 로컬 `run_carry_input_ablation.py --reuse-training <결과폴더>`로 동일 물리 평가를 수행한다.
학습 적합도와 운반 성공률은 별개다. 시뮬레이션 이전·Drive 동기화·GPU 구매·자동 재접속은 포함하지 않는다.
